# Décider et mesurer

Deux scores : quel verdict, et à quel prix ?

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/secumail")
except ImportError:
    os.chdir(Path.cwd().parents[1])  # depuis workshop/<atelier>/

sys.path.insert(0, os.getcwd())

Les deux modules produisent chacun un score. Le **moteur de scoring** tranche.

# La chaîne de décision

<svg viewBox="0 0 760 330" width="760" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="17">
  <defs><marker id="f" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#444"/></marker></defs>
  <g fill="none" stroke="#444" stroke-width="1.6" marker-end="url(#f)">
    <path d="M380,48 L200,88"/><path d="M380,48 L560,88"/>
    <path d="M200,150 L360,196"/><path d="M560,150 L400,196"/>
    <path d="M380,244 L110,284"/><path d="M380,244 L290,284"/><path d="M380,244 L470,284"/><path d="M380,244 L650,284"/>
  </g>
  <g fill="#eef3fb" stroke="#1f5fa8" stroke-width="1.6">
    <rect x="300" y="8" width="160" height="40" rx="6"/>
    <rect x="85" y="90" width="230" height="60" rx="6"/><rect x="445" y="90" width="230" height="60" rx="6"/>
    <rect x="280" y="198" width="200" height="46" rx="6"/>
    <rect x="50" y="286" width="120" height="38" rx="6"/><rect x="230" y="286" width="120" height="38" rx="6"/>
    <rect x="590" y="286" width="120" height="38" rx="6"/>
  </g>
  <rect x="410" y="286" width="120" height="38" rx="6" fill="#fff" stroke="#b0313f" stroke-width="2" stroke-dasharray="6 4"/>
  <g text-anchor="middle" fill="#17202b">
    <text x="380" y="34">Courriel reçu</text>
    <text x="200" y="115" font-weight="bold">Module 1 · Hameçonnage</text><text x="200" y="138">« Qui écrit ? »</text>
    <text x="560" y="115" font-weight="bold">Module 2 · Spam</text><text x="560" y="138">« Que dit-on ? »</text>
    <text x="380" y="227" font-weight="bold">Moteur de scoring</text>
    <text x="110" y="311">HAM</text><text x="290" y="311">SPAM</text><text x="470" y="311" fill="#b0313f">?</text><text x="650" y="311">PHISHING</text>
  </g>
  <g fill="#5a6573" font-size="14"><text x="225" y="182">score_phishing</text><text x="470" y="182">score_spam</text></g>
</svg>

**Il manque un verdict. Lequel ? Quand mène-t-on un courriel là ?** (exercice 1.1)

## Les actions du moteur

In [ ]:
from src.scoring_engine import ACTIONS

for verdict, action in ACTIONS.items():
    print(f"{verdict:<9} -> {action}")

## Les seuils

Le moteur compare chaque score à des **seuils**, toujours dans cet ordre :

<img src="images/arbre_seuils.svg" alt="" style="width:600px">

## Essayons

In [ ]:
from homework import thresholds
from src.scoring_engine import decider

print(thresholds.SEUIL_PHISHING, thresholds.SEUIL_PHISHING_SUSPECT)
print(thresholds.SEUIL_SPAM, thresholds.SEUIL_SPAM_SUSPECT)
paires = [(60, 10), (30, 50), (10, 45), (10, 25), (0, 0)]
for score_phishing, score_spam in paires:
    verdict = decider(score_phishing, score_spam)
    print(f"phishing={score_phishing:>3}  spam={score_spam:>3}  ->  {verdict}")

**Le deuxième courriel a un score de spam de 50. Pourquoi n'est-il pas classé SPAM ?** (exercice 1.2)

## La règle d'arbitrage

Quand les deux scores sont élevés, l'hameçonnage l'emporte.
Les conséquences ne sont pas comparables : une publicité livrée par erreur agace,
un vol d'identifiants compromet l'organisation. On teste donc le phishing **en premier**.

## Ramener un score sur 100

La somme brute des règles peut dépasser 100. Le module 1 peut monter jusqu'à 130.

| Stratégie | Formule | Effet |
|---|---|---|
| **Plafonner** | `min(score, 100)` | conserve les petits écarts |
| **Normaliser** | `score / max_theorique * 100` | écrase tout vers le bas |

In [ ]:
from src.scoring_engine import borner

print(borner(130), borner(45), borner(-5))

Le moteur plafonne. Ce n'est pas un choix neutre.

# Un courriel à la main

Détachez les figures à la fin du cahier. Relevez chaque indice et ses points,
calculez les deux scores, puis donnez le verdict avec les seuils. (exercice 4.1)

Vérifiez ensuite avec la section 3 de `main.ipynb` : vos règles trouvent-elles les mêmes indices que vous ?

## Le piège de l'exactitude

Dans une vraie boîte, 96 courriels sur 100 sont légitimes.
**Quelle note obtient ce gardien ?** (exercice 4.3)

<img src="images/gardien_exactitude.svg" alt="" style="width:760px">

## Calculons

In [ ]:
from src import evaluation

reels = ["ham"] * 96 + ["phishing"] * 4
toujours_ham = ["ham"] * 100

bonnes = sum(r == v for r, v in zip(reels, toujours_ham))
exactitude = bonnes / len(reels)
resultat = evaluation.mesures(reels, toujours_ham)
print(f"Exactitude               : {exactitude:.0%}")
rappel = resultat["rappel_phishing_prudente"]
print(f"Hameçonnages trouvés     : {rappel:.0%}")
print(f"Coût total pondéré       : {resultat['cout']}")

96 % de bonnes réponses, et **aucun** hameçonnage arrêté. L'exactitude est interdite comme mesure principale.

## Toutes les erreurs ne se valent pas

In [ ]:
couts = sorted(evaluation.COUTS.items(), key=lambda x: -x[1])
for (reel, verdict), cout in couts:
    print(f"{reel:>9} classé {verdict:<9} coûte {cout}")

Bloquer un courriel légitime (10) et laisser passer un hameçonnage (8) : aucun réglage ne minimise les deux à la fois.
**Êtes-vous d'accord avec ces coûts ? Lequel changeriez-vous ?** (exercice 4.2)

## Et la quarantaine ?

Les étiquettes n'ont que trois valeurs : `ham`, `spam`, `phishing`. Le système peut aussi répondre `suspect`.

In [ ]:
verdicts = ["ham", "suspect", "phishing", "suspect"]
for politique in ["prudente", "permissive"]:
    convertis = evaluation.appliquer_politique(verdicts, politique)
    print(f"{politique:<10} : {convertis}")

Les mêmes verdicts donnent deux résultats. La performance dépend de ce qu'on fait des **cas incertains**.

## Lire la matrice de confusion

<img src="images/matrice.svg" alt="" style="width:560px">

## L'ancienne passerelle, qui répond au hasard

In [ ]:
from src import pipeline, scoring_engine

courriels = pipeline.charger_courriels()
_, validation = pipeline.separer_validation(courriels)
reels = [c["label"] for c in validation]
verdicts = scoring_engine.ancienne_passerelle(validation)
evaluation.matrice_confusion(reels, verdicts)

# À vous : régler le système

Un seuil choisi au hasard puis jamais revu est une faute de méthode.

- Changez **une seule** valeur à la fois : un `SEUIL_` ou des `POINTS_`.
- Mesurez, et notez le coût avant et après. (exercice 6.1)
- Gardez le changement seulement si le coût baisse.

## Le coût selon un seuil

La section 7 de `main.ipynb` trace une courbe par seuil. Exemple avec les règles de départ seulement :

In [ ]:
%matplotlib inline
table_marques = pipeline.charger_domaines_officiels()
resultats = scoring_engine.analyser_tous(validation, table_marques)
pipeline.tracer_cout_selon_seuil(resultats, reels, "SEUIL_SPAM")